# Web Scraping con Python

Este notebook demuestra cómo extraer datos directamente desde páginas HTML públicas usando **`requests`** y **`BeautifulSoup`**.

| Sitio | Datos extraídos |
|-------|-----------------|
| **Books to Scrape** | Libros, precios y ratings (sitio de práctica) |
| **Quotes to Scrape** | Frases célebres y autores (sitio de práctica) |

> Ambos sitios son **sandboxes públicos creados para practicar scraping**, sin restricciones ni autenticación.

**Tecnologías:** `requests`, `beautifulsoup4`, `pandas`, `pyarrow`

## 1. Instalación de Dependencias

In [2]:
%pip install requests beautifulsoup4 pandas pyarrow lxml

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)

   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importación de Librerías

In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

DIR_OUTPUT = Path('datos/output')
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

print('Librerías importadas correctamente.')
print(f'Directorio de salida: {DIR_OUTPUT.resolve()}')

Librerías importadas correctamente.
Directorio de salida: C:\Users\Sergio Orozco\Downloads\clases\Ingenieria_datos\notebook\unidad_II\datos\output


---
## PARTE A — Books to Scrape

URL: http://books.toscrape.com  
Sitio de práctica con catálogo de libros ficticios.

### A1. Petición GET y HTML Crudo

In [4]:
# Petición GET a la página principal
url_books = 'http://books.toscrape.com/catalogue/page-1.html'
respuesta = requests.get(url_books)

# Ver la respuesta cruda
print('Status code :', respuesta.status_code)
print('Content-Type:', respuesta.headers['Content-Type'])
print('\nHTML raw (primeros 1000 caracteres):')
print(respuesta.text[:1000])

Status code : 200
Content-Type: text/html

HTML raw (primeros 1000 caracteres):


<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:30" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![end

### A2. Parsear el HTML con BeautifulSoup

In [5]:
# Parsear el HTML
soup = BeautifulSoup(respuesta.text, 'lxml')

# Ver el título de la página
print('Título de la página:', soup.title.text.strip())

# Ver cuántos artículos (libros) hay en esta página
articulos = soup.select('article.product_pod')
print(f'Libros encontrados en esta página: {len(articulos)}')

# Ver el HTML de un solo artículo para entender su estructura
print('\nEstructura HTML del primer libro:')
print(articulos[0].prettify())

Título de la página: All products | Books to Scrape - Sandbox
Libros encontrados en esta página: 20

Estructura HTML del primer libro:
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
<

### A3. Extraer Datos de una Página

In [6]:
# Mapeo de palabras a números para el rating
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

libros = []
for art in articulos:
    titulo  = art.h3.a['title']
    precio  = art.select_one('p.price_color').text.strip()
    rating  = rating_map.get(art.p['class'][1], 0)
    stock   = art.select_one('p.availability').text.strip()
    libros.append({'titulo': titulo, 'precio': precio, 'rating': rating, 'disponibilidad': stock})

df_libros_p1 = pd.DataFrame(libros)
print(f'Libros extraídos (página 1): {len(df_libros_p1)}')
df_libros_p1

Libros extraídos (página 1): 20


,titulo,precio,rating,disponibilidad
0,A Light in the Attic,Â£51.77,3,In stock
1,Tipping the Velvet,Â£53.74,1,In stock
2,Soumission,Â£50.10,1,In stock
3,Sharp Objects,Â£47.82,4,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,5,In stock
5,The Requiem Red,Â£22.65,1,In stock
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,4,In stock
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,3,In stock
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,4,In stock
9,The Black Maria,Â£52.15,1,In stock


### A4. Scraping de Múltiples Páginas (primeras 5)

In [7]:
BASE_URL   = 'http://books.toscrape.com/catalogue/page-{}.html'
PAGINAS    = 5
todos      = []

for num_pagina in range(1, PAGINAS + 1):
    url   = BASE_URL.format(num_pagina)
    resp  = requests.get(url)
    s     = BeautifulSoup(resp.text, 'lxml')
    arts  = s.select('article.product_pod')

    for art in arts:
        todos.append({
            'pagina':         num_pagina,
            'titulo':         art.h3.a['title'],
            'precio':         art.select_one('p.price_color').text.strip(),
            'rating':         rating_map.get(art.p['class'][1], 0),
            'disponibilidad': art.select_one('p.availability').text.strip(),
        })
    print(f'  Página {num_pagina}: {len(arts)} libros — status {resp.status_code}')

df_libros = pd.DataFrame(todos)

# Limpiar el precio y convertir a float
df_libros['precio_gbp'] = df_libros['precio'].str.replace('Â£', '').str.replace('£', '').astype(float)

print(f'\nTotal de libros extraídos: {len(df_libros)}')
df_libros.head(10)

  Página 1: 20 libros — status 200
  Página 2: 20 libros — status 200
  Página 3: 20 libros — status 200
  Página 4: 20 libros — status 200
  Página 5: 20 libros — status 200

Total de libros extraídos: 100


,pagina,titulo,precio,rating,disponibilidad,precio_gbp
0,1,A Light in the Attic,Â£51.77,3,In stock,51.77
1,1,Tipping the Velvet,Â£53.74,1,In stock,53.74
2,1,Soumission,Â£50.10,1,In stock,50.10
3,1,Sharp Objects,Â£47.82,4,In stock,47.82
4,1,Sapiens: A Brief History of Humankind,Â£54.23,5,In stock,54.23
5,1,The Requiem Red,Â£22.65,1,In stock,22.65
6,1,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,4,In stock,33.34
7,1,The Coming Woman: A Novel Based on the Life of...,Â£17.93,3,In stock,17.93
8,1,The Boys in the Boat: Nine Americans and Their...,Â£22.60,4,In stock,22.60
9,1,The Black Maria,Â£52.15,1,In stock,52.15


### A5. Análisis Rápido

In [8]:
print('=== Estadísticas de precio ===')
print(df_libros['precio_gbp'].describe().round(2))

print('\n=== Distribución por rating ===')
df_libros.groupby('rating')[['titulo']].count().rename(columns={'titulo': 'cantidad'})

=== Estadísticas de precio ===
count    100.00
mean      34.56
std       14.64
min       10.16
25%       19.90
50%       34.78
75%       47.97
max       58.11
Name: precio_gbp, dtype: float64

=== Distribución por rating ===


,cantidad
rating,
1,22
2,19
3,22
4,18
5,19


### A6. Guardar en CSV y Parquet

In [9]:
df_libros.to_csv(DIR_OUTPUT / 'books_toscrape.csv', index=False, encoding='utf-8-sig')
df_libros.to_parquet(DIR_OUTPUT / 'books_toscrape.parquet', index=False, engine='pyarrow')

print(f'✅ CSV     : {DIR_OUTPUT / "books_toscrape.csv"}')
print(f'✅ Parquet : {DIR_OUTPUT / "books_toscrape.parquet"}')

✅ CSV     : datos\output\books_toscrape.csv
✅ Parquet : datos\output\books_toscrape.parquet


---
## PARTE B — Quotes to Scrape

URL: http://quotes.toscrape.com  
Sitio de práctica con frases célebres y datos de autores.

### B1. Petición GET y HTML Crudo

In [10]:
# Petición GET a la página principal
url_quotes = 'http://quotes.toscrape.com/'
respuesta_q = requests.get(url_quotes)

# Ver la respuesta cruda
print('Status code :', respuesta_q.status_code)
print('Content-Type:', respuesta_q.headers['Content-Type'])
print('\nHTML raw (primeros 1200 caracteres):')
print(respuesta_q.text[:1200])

Status code : 200
Content-Type: text/html; charset=utf-8

HTML raw (primeros 1200 caracteres):
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. 

### B2. Parsear y Ver Estructura

In [11]:
soup_q = BeautifulSoup(respuesta_q.text, 'lxml')

# Ver el título
print('Título:', soup_q.title.text.strip())

# Ver cuántas frases hay en esta página
quotes = soup_q.select('div.quote')
print(f'Frases en esta página: {len(quotes)}')

# Ver la estructura HTML de la primera frase
print('\nEstructura de la primera frase:')
print(quotes[0].prettify())

Título: Quotes to Scrape
Frases en esta página: 10

Estructura de la primera frase:
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
  <a class="tag" href="/tag/change/page/1/">
   change
  </a>
  <a class="tag" href="/tag/deep-thoughts/page/1/">
   deep-thoughts
  </a>
  <a class="tag" href="/tag/thinking/page/1/">
   thinking
  </a>
  <a class="tag" href="/tag/world/page/1/">
   world
  </a>
 </div>
</div>



### B3. Scraping de Todas las Páginas

In [12]:
BASE_QUOTES = 'http://quotes.toscrape.com/page/{}/' 
frases      = []
pagina      = 1

while True:
    resp  = requests.get(BASE_QUOTES.format(pagina))
    s     = BeautifulSoup(resp.text, 'lxml')
    items = s.select('div.quote')

    if not items:
        print(f'  Página {pagina}: sin resultados — fin del scraping.')
        break

    for item in items:
        frases.append({
            'frase':  item.select_one('span.text').text.strip('\u201c\u201d'),
            'autor':  item.select_one('small.author').text.strip(),
            'tags':   ', '.join(tag.text for tag in item.select('a.tag')),
        })

    print(f'  Página {pagina}: {len(items)} frases — status {resp.status_code}')

    # Verificar si existe botón "Next"
    if not s.select_one('li.next'):
        break
    pagina += 1

df_quotes = pd.DataFrame(frases)
print(f'\nTotal de frases extraídas: {len(df_quotes)}')
df_quotes

  Página 1: 10 frases — status 200
  Página 2: 10 frases — status 200
  Página 3: 10 frases — status 200
  Página 4: 10 frases — status 200
  Página 5: 10 frases — status 200
  Página 6: 10 frases — status 200
  Página 7: 10 frases — status 200
  Página 8: 10 frases — status 200
  Página 9: 10 frases — status 200
  Página 10: 10 frases — status 200

Total de frases extraídas: 100


,frase,autor,tags
0,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices"
2,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor"
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational"
...,...,...,...
95,You never really understand a person until you...,Harper Lee,better-life-empathy
96,You have to write the book that wants to be wr...,Madeleine L'Engle,"books, children, difficult, grown-ups, write, ..."
97,Never tell the truth to people who are not wor...,Mark Twain,truth
98,"A person's a person, no matter how small.",Dr. Seuss,inspirational


### B4. Frases más Frecuentes por Autor

In [13]:
print('=== Frases por autor ===')
df_quotes.groupby('autor')[['frase']].count().rename(columns={'frase': 'cantidad'}).sort_values('cantidad', ascending=False)

=== Frases por autor ===


,cantidad
autor,
Albert Einstein,10
J.K. Rowling,9
Marilyn Monroe,7
Mark Twain,6
Dr. Seuss,6
C.S. Lewis,5
Jane Austen,5
Bob Marley,3
Eleanor Roosevelt,2


### B5. Guardar en CSV y Parquet

In [14]:
df_quotes.to_csv(DIR_OUTPUT / 'quotes_toscrape.csv', index=False, encoding='utf-8-sig')
df_quotes.to_parquet(DIR_OUTPUT / 'quotes_toscrape.parquet', index=False, engine='pyarrow')

print(f'✅ CSV     : {DIR_OUTPUT / "quotes_toscrape.csv"}')
print(f'✅ Parquet : {DIR_OUTPUT / "quotes_toscrape.parquet"}')

✅ CSV     : datos\output\quotes_toscrape.csv
✅ Parquet : datos\output\quotes_toscrape.parquet


---
## PARTE C — Resumen de Archivos Generados

In [15]:
archivos = [
    {'archivo': f.name, 'formato': f.suffix, 'tamaño_KB': round(f.stat().st_size / 1024, 2)}
    for f in sorted(DIR_OUTPUT.iterdir())
]
print('Archivos en datos/output:')
pd.DataFrame(archivos)

Archivos en datos/output:


,archivo,formato,tamaño_KB
0,books_toscrape.csv,.csv,7.63
1,books_toscrape.parquet,.parquet,9.36
2,clima_buenosaires_7dias.csv,.csv,0.27
3,clima_buenosaires_7dias.parquet,.parquet,3.73
4,clima_buenosaires_hist_2026-03-21_2026-04-20.csv,.csv,0.84
5,clima_buenosaires_hist_2026-03-21_2026-04-20.p...,.parquet,3.58
6,feriados_AR_2026.csv,.csv,1.61
7,feriados_AR_2026.parquet,.parquet,6.09
8,person_person.csv,.csv,917.14
9,person_person.parquet,.parquet,241.73
